In [ ]:
import os
import uuid
import hashlib
import platform
import numpy as np
import scipy
import matplotlib
import pandas as pd

# 修改为自己的学号
STUDENT_ID = 20244080220

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()} | {platform.machine()} | {platform.system()} | {uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID} - {MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


## 实验1：NumPy 数组、数据类型与索引

In [ ]:
# 1.1 整数、浮点数、布尔值 ndarray
arr_int = np.array([1, 2, 3, 4])
arr_float = np.array([1.0, 2.0, 3.0])
arr_bool = np.array([True, False, True])
for name, arr in [("int", arr_int), ("float", arr_float), ("bool", arr_bool)]:
    print(name, "dtype =", arr.dtype, "shape =", arr.shape, "dim =", arr.ndim)


NumPy 数组是一块连续的同质数据，所有元素共享同一个 dtype，内存紧凑、运算快；
Python 列表可以混存不同类型的对象，本质是指针数组，灵活性高但效率低。

In [ ]:
# 1.2 改变形状
a = np.arange(12)
print("原数组:", a)
a34 = a.reshape(3, 4)
print("(3,4):\n", a34)
a232 = a.reshape(2, 3, 2)
print("(2,3,2):\n", a232)
# 改变形状前后元素总数必须相等：3*4=12，2*3*2=12


In [ ]:
# 1.3 索引
b = np.arange(12).reshape(3, 4)
print("b =\n", b)
row2 = b[1, :]
print("第2行:", row2, "shape =", row2.shape)
col13 = b[:, 0:3]
print("第1至3列:\n", col13, "shape =", col13.shape)
gt6 = b[b > 6]
print("大于6的元素:", gt6, "shape =", gt6.shape)


In [ ]:
# 1.4 布尔筛选
c = np.arange(10)
sel1 = c[(c < 2) | (c > 7)]
print("小于2或大于7:", sel1)
sel2 = c[(c > 3) & (c < 7)]
print("大于3且小于7:", sel2)


In [ ]:
# 1.5 逐元素乘法 vs 矩阵乘法
m1 = np.array([[1, 2], [3, 4]])
m2 = np.array([[5, 6], [7, 8]])
print("逐元素乘法 *:\n", m1 * m2)
print("np.multiply:\n", np.multiply(m1, m2))
print("矩阵乘法 @:\n", m1 @ m2)
print("np.matmul:\n", np.matmul(m1, m2))


`*` 和 `np.multiply` 是对应位置逐个相乘；`@` 和 `np.matmul` 是矩阵乘法，
结果的每个元素是一行与一列的内积。

## 实验2：随机抽样、统计与缺失值

In [ ]:
# 2.1 模拟成绩
rng = np.random.default_rng(42)
scores = rng.normal(70, 10, 1000)
scores = np.clip(scores, 0, 100)
print("样本数:", scores.size)
print("最小值:", scores.min(), "最大值:", scores.max())
print("均值:", scores.mean(), "标准差:", scores.std())
print("极差:", scores.max() - scores.min())


In [ ]:
# 2.2 不放回抽样
pop = np.arange(1, 11)
uniform = rng.choice(pop, size=5, replace=False)
weighted = rng.choice(pop, size=5, replace=False, p=pop / pop.sum())
print("均匀抽样:", uniform)
print("按大小加权抽样:", weighted)


In [ ]:
# 2.3 排列：保留原数组 vs 原地打乱
orig = np.arange(10)
shuffled_copy = rng.permutation(orig)
print("permutation 后原数组:", orig)
print("返回的新数组:", shuffled_copy)
orig2 = np.arange(10)
rng.shuffle(orig2)
print("shuffle 后原数组:", orig2)


In [ ]:
# 2.4 缺失值
scores_nan = scores.copy()
scores_nan[[10, 20, 30]] = np.nan
print("普通均值:", scores_nan.mean())
print("忽略缺失值的均值:", np.nanmean(scores_nan))
print("缺失值个数:", np.isnan(scores_nan).sum())


只要数组里含 np.nan，普通统计函数（如 mean）就会返回 nan；
需要用 nanmean 等忽略缺失值的函数，或先去掉缺失值再统计。

In [ ]:
# 2.5 内存表格变量（不写入磁盘）
score_table = pd.DataFrame({
    "index": np.arange(len(scores_nan)),
    "score": scores_nan,
})
print(score_table.head())


## 实验3：SciPy 数值计算与 Matplotlib 可视化

In [ ]:
# 3.1 解线性方程组 3x + y = 9, x + 2y = 8
from scipy.linalg import solve
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])
x = solve(A, b)
print("解:", x)
print("代入验证 A@x =", A @ x)


In [ ]:
# 3.2 求方程 x^2 - 2 = 0 的正根和负根
from scipy.optimize import root
def f(x):
    return x ** 2 - 2

res_pos = root(f, 1)
res_neg = root(f, -1)
print("正根:", res_pos.x, "success:", res_pos.success, "函数值:", f(res_pos.x))
print("负根:", res_neg.x, "success:", res_neg.success, "函数值:", f(res_neg.x))


In [ ]:
# 3.3 求 f(x) = x^2 + x + 2 的最小值
from scipy.optimize import minimize
def g(x):
    return x ** 2 + x + 2

res = minimize(g, 0)
print("极小点:", res.x, "函数值:", res.fun)
print("解析结果：极小点 x = -0.5，函数值 = 1.75")


In [ ]:
# 3.4 独立样本 t 检验
from scipy import stats
group1 = rng.normal(0, 1, 100)
group2 = rng.normal(1, 1, 100)
t_stat, p_value = stats.ttest_ind(group1, group2)
print("t 统计量:", t_stat, "p 值:", p_value)
print("0.05 水平判断:", "拒绝原假设，两组均值有显著差异" if p_value < 0.05 else "不能拒绝原假设")


In [ ]:
# 3.5 可视化
import matplotlib.pyplot as plt

valid_scores = scores_nan[~np.isnan(scores_nan)]
plt.figure()
plt.hist(valid_scores, bins=20, edgecolor="black")
plt.title("有效成绩直方图")
plt.xlabel("成绩")
plt.ylabel("频数")
plt.show()

plt.figure()
plt.boxplot(valid_scores)
plt.title("有效成绩箱线图")
plt.ylabel("成绩")
plt.show()

x = np.linspace(0, 5, 100)
plt.figure()
plt.plot(x, x, label="y = x")
plt.plot(x, x ** 2, label="y = x²")
plt.plot(x, x ** 3, label="y = x³")
plt.title("函数折线图")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()


## 实验4：pandas Series、DataFrame 与内存中的 CSV 读写

In [ ]:
# 4.1 创建 DataFrame
records = {
    "Name": ["Alice", "Bob", "Carol", "David", "Eva", "Frank"],
    "Group": ["A", "A", "B", "B", "A", "B"],
    "Math": [88, 72, 91, 65, 79, 84],
    "English": [90, 68, 85, 73, 92, 77],
    "Python": [95, 80, 89, 70, 86, 82],
}
df = pd.DataFrame(records).set_index("Name")
subjects = ["Math", "English", "Python"]
df["Total"] = df[subjects].sum(axis=1)
df["Average"] = df[subjects].mean(axis=1)
print("行索引:", list(df.index))
print("列名:", list(df.columns))
print(df.head(3))
print(df.describe())


In [ ]:
# 4.2 loc / iloc 读取与筛选
print("loc Carol:\n", df.loc["Carol"])
print("iloc 第3行:\n", df.iloc[2])
print("平均分不低于80:\n", df[df["Average"] >= 80])


In [ ]:
# 4.3 列均值与行均值
print("列均值 mean(axis=0):\n", df[subjects].mean(axis=0))
print("行均值 mean(axis=1):\n", df[subjects].mean(axis=1))


In [ ]:
# 4.4 apply / agg / rank
df["Max"] = df[subjects].apply(lambda row: row.max(), axis=1)
print("每名学生三科最高分:\n", df["Max"])
print("agg 每科 min/mean/max:\n", df[subjects].agg(["min", "mean", "max"]))
print("Python 成绩排名（降序）:\n", df["Python"].rank(ascending=False))


In [ ]:
# 4.5 分组统计与 transform
print("每组人数:\n", df.groupby("Group").size())
print("三科组均值:\n", df.groupby("Group")[subjects].mean())
df["Group_Python_Mean"] = df.groupby("Group")["Python"].transform("mean")
print(df)


In [ ]:
# 4.6 filter 保留 Python 平均分不低于 80 的组
keep = df.groupby("Group").filter(lambda g: g["Python"].mean() >= 80)
print("保留的组和学生:\n", keep)


In [ ]:
# 4.7 协方差矩阵与相关系数矩阵
cov = df[subjects].cov()
corr = df[subjects].corr()
print("协方差矩阵:\n", cov)
print("相关系数矩阵:\n", corr)


协方差为正，表示两科成绩同增同减；为负则此消彼长。相关系数介于 -1 到 1 之间，
绝对值越接近 1，线性相关性越强；接近 0 说明线性关系很弱。